# Acoustic mask listening test

Run the setup cell, then edit the selection and run the playback cell. You can open this notebook from the repository root or from `notebooks/`. Select the Python kernel where the repository is installed (`python -m pip install -e .`).

Start at low speaker or headphone volume. The carrier-only clip and the mixture use the exact same excerpt and a shared playback gain, with player normalization disabled. A mask-only clip uses that same gain, so it will usually be quieter. Keep your device volume fixed during a comparison.

In [ ]:
from functools import lru_cache
from pathlib import Path

import librosa
import numpy as np
from IPython.display import Audio, display

from acoustic_masking.audio import highpass_audio, normalize_source, rms, split_carrier_audio
from acoustic_masking.config import N_SAMPLES, SAMPLE_RATE, read_config

root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "configs/evaluation.yaml").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Start Jupyter within the repository.")

bank_path = root / "artifacts/masks/candidate_bank.npz"
with np.load(bank_path, allow_pickle=False) as bank:
    masks = {name: bank[name].astype(np.float32) for name in sorted(bank.files)}
if not masks or any(
    wave.shape != (N_SAMPLES,) or not np.isfinite(wave).all() for wave in masks.values()
):
    raise ValueError("The bank must contain finite, 0.25-second mono masks.")

settings = read_config(root / "configs/evaluation.yaml")
print(f"Loaded {len(masks)} masks at {SAMPLE_RATE:,} Hz.")
for mode in ("zero", "random"):
    print(f"{mode}: " + ", ".join(name for name in masks if name.startswith(f"{mode}_")))

## Select a mask and carrier

Try the same start time and shared playback gain across several masks. `mask_gain = 1.0` uses the experiment's playback scale and carrier-specific residual gain. Values of `0.90`, `0.75`, and `0.50` reduce the residual by about 0.9, 2.5, and 6 dB. Start with `carrier_part = "full"` for longer excerpts, or use `"val"` to match the evaluation split and its RMS normalization. Clips shorten automatically near the end of the chosen recording segment.

The residual repeats every 0.25 seconds. Listen for buzzing, ticking, or a regular rhythm at the loop boundary. The mixture contains environmental audio and the mask, with no keystrokes.

In [ ]:
mask_name = "zero_1101"   # Pick any key printed above.
carrier_name = "rain"      # "rain", "fan", or "wind"
carrier_part = "full"     # "full", "train", "val", or "test"
start_seconds = 0.0       # Relative to the selected recording segment.
duration_seconds = 5.0
phase_seconds = 0.0       # Offset within the repeating residual.
mask_gain = 0.8
playback_gain = 8.0        # Shared listening boost for all clips.
blind = False             # True shows randomized A/B labels.
blind_seed = 40           # Change this for a fresh A/B assignment.

In [ ]:
@lru_cache(maxsize=12)
def get_carrier(name, part):
    source, _ = librosa.load(root / settings["carriers"][name], sr=SAMPLE_RATE, mono=True)
    source = highpass_audio(source)
    if part != "full":
        source = split_carrier_audio(source)[part]
    return normalize_source(source, settings["carrier_config"][name]["carrier_rms"])


def make_clips(mask, carrier, part, start, duration, phase, gain):
    if not np.isfinite([start, duration, phase, gain]).all() or start < 0 or duration <= 0 or gain < 0:
        raise ValueError("Use a nonnegative start and mask gain and a positive duration.")
    source = get_carrier(carrier, part)
    first = round(start * SAMPLE_RATE)
    count = round(duration * SAMPLE_RATE)
    excerpt = source[first:first + count]
    if not excerpt.size:
        raise ValueError(f"Choose a start within the {len(source) / SAMPLE_RATE:.2f} s recording segment.")
    phase_samples = round(phase * SAMPLE_RATE)
    indices = (np.arange(excerpt.size) - phase_samples) % N_SAMPLES
    residual_gain = settings["playback_scale"] * settings["carrier_config"][carrier]["residual_gain"]
    residual = masks[mask][indices] * (gain * residual_gain)
    return {"Carrier only": excerpt, "Carrier + mask": excerpt + residual, "Mask only": residual}


clips = make_clips(
    mask_name, carrier_name, carrier_part, start_seconds, duration_seconds, phase_seconds, mask_gain
)
if not np.isfinite(playback_gain) or playback_gain <= 0:
    raise ValueError("Playback gain must be positive and finite.")
peak_after_gain = max(float(np.max(np.abs(clip))) for clip in clips.values()) * playback_gain
if peak_after_gain > 1:
    raise ValueError("Playback would clip. Reduce the shared playback gain and rerun.")

comparison = ["Carrier only", "Carrier + mask"]
if blind:
    np.random.default_rng(blind_seed).shuffle(comparison)
    assignment = dict(zip(("A", "B"), comparison))
else:
    assignment = {name: name for name in [*comparison, "Mask only"]}
    relative_db = 20 * np.log10(rms(clips["Mask only"]) / rms(clips["Carrier only"])) if mask_gain else -np.inf
    print(f"Residual-to-carrier RMS in this excerpt: {relative_db:.1f} dB.")
print(f"Excerpt length: {len(clips['Carrier only']) / SAMPLE_RATE:.2f} s.")
for label, name in assignment.items():
    print(label)
    display(Audio(clips[name] * playback_gain, rate=SAMPLE_RATE, normalize=False))

## Record an impression before revealing A/B

For a blind trial, set `blind = True` and play A and B in alternating order. Record which clip you think contains the mask, whether you heard a recurring artifact, and how annoying each clip was from 0 (none) to 4 (very annoying). Try another excerpt, carrier, or mask gain while keeping device volume fixed. Run the next cell after recording your impression to reveal the assignment.

In [ ]:
if blind:
    for label, name in assignment.items():
        print(f"{label} = {name}")
else:
    print("The players are already labeled.")